In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import pandas as pd
from pandas_datareader import data as pdr

# -----------------------
# Config
# -----------------------
TICKERS = ["SPY", "QQQ", "IWM", "EEM", "TLT"]
START = "2008-01-01"
END   = "2026-03-06"

# -----------------------
# 1) Prices from manually downloaded Stooq CSVs
# -----------------------
def read_stooq_csv(ticker):
    path = f"../data/{ticker.lower()}_us_d.csv"

    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.sort_values("Date").set_index("Date")

    return df["Close"].rename(ticker)


close = pd.concat(
    [read_stooq_csv(t) for t in TICKERS],
    axis=1
)

close = close.dropna(how="all").ffill()
close = close[TICKERS]

# optionally enforce START/END after loading manual CSVs
close = close.loc[pd.Timestamp(START):]
if END is not None:
    close = close.loc[:pd.Timestamp(END)]


# -----------------------
# 2) VIX from FRED
# -----------------------
vix = pd.read_csv(
    "../data/VIX.csv",
    parse_dates=["Date"],
    index_col="Date"
)

vix = vix.sort_index()
print(vix.columns)
print([repr(c) for c in vix.columns])
vix = vix[[" Close"]].rename(columns={" Close": "VIX"})
vix["VIX"] = pd.to_numeric(vix["VIX"], errors="coerce")
vix = vix.ffill()

print(vix.head())

# align
idx = close.index.intersection(vix.index)
close = close.loc[idx]
vix = vix.loc[idx]

# -----------------------
# 3) Volatility target y_t = |log return|
# -----------------------
logp = np.log(close)
rets = logp.diff()
vol = rets.abs().dropna()

vix = vix.loc[vol.index]
vix = vix.shift(1).ffill().dropna()
print(vol.shape, vix.shape)
print(vol.head())
print(vix.head())

In [ ]:
# 1) align both to a business-day index first
idx_b = pd.date_range(close.index.min(), close.index.max(), freq="B")

close_b = close.reindex(idx_b).ffill()   # carry last close through holidays/weekends
vix_b   = vix.reindex(idx_b).ffill()

# 2) compute vol on this regular grid
logp = np.log(close_b)
rets = logp.diff()
vol  = rets.abs().iloc[1:]              # drop first nan
vix  = vix_b.loc[vol.index]

In [ ]:
from darts import TimeSeries

def to_ts_from_dfcol(df_col: pd.Series, name: str):
    s = df_col.dropna().sort_index()
    # should already be tz-naive; keep it that way
    return TimeSeries.from_times_and_values(
        s.index,
        s.values.astype(np.float32),
        columns=[name],
    )

series_list = [to_ts_from_dfcol(vol[c], c) for c in vol.columns]
vix_ts = to_ts_from_dfcol(vix["VIX"], "VIX")

print("Example series:", series_list[0].start_time(), "->", series_list[0].end_time(), "len=", len(series_list[0]))
print("VIX:", vix_ts.start_time(), "->", vix_ts.end_time(), "len=", len(vix_ts))

In [ ]:
TRAIN_END = pd.Timestamp("2018-12-31")
VAL_END   = pd.Timestamp("2019-12-31")

def split_ts(ts: TimeSeries, train_end, val_end):
    idx = ts.time_index
    va_start = idx[idx > train_end][0]
    te_start = idx[idx > val_end][0]
    tr = ts.slice(idx[0], train_end)
    va = ts.slice(va_start, val_end)
    te = ts.slice(te_start, idx[-1])
    return tr, va, te

train_raw, val_raw, test_raw = [], [], []
for ts in series_list:
    tr, va, te = split_ts(ts, TRAIN_END, VAL_END)
    train_raw.append(tr); val_raw.append(va); test_raw.append(te)

train_vix, val_vix, test_vix = split_ts(vix_ts, TRAIN_END, VAL_END)

print("Lens:", len(train_raw[0]), len(val_raw[0]), len(test_raw[0]))

In [ ]:
def ts_values(ts):
    return ts.values(copy=False).squeeze(-1).astype(np.float64)

def zscore_lists(train_raw, val_raw, test_raw, eps=1e-8):
    train_z, val_z, test_z, stats = [], [], [], []
    for tr, va, te in zip(train_raw, val_raw, test_raw):
        tr_v = ts_values(tr)
        mu = float(np.nanmean(tr_v))
        sd = float(np.nanstd(tr_v, ddof=0))
        sd = sd if np.isfinite(sd) and sd > eps else 1.0
        stats.append((mu, sd))

        def norm(ts):
            v = ts_values(ts)
            v = (v - mu) / sd
            return TimeSeries.from_times_and_values(ts.time_index, v.astype(np.float32))

        train_z.append(norm(tr)); val_z.append(norm(va)); test_z.append(norm(te))
    return train_z, val_z, test_z, stats

train, val, test, zstats = zscore_lists(train_raw, val_raw, test_raw)

# z-score VIX using train stats (single series)
vtr = ts_values(train_vix)
vix_mu = float(np.nanmean(vtr))
vix_sd = float(np.nanstd(vtr, ddof=0))
vix_sd = vix_sd if vix_sd > 1e-8 else 1.0

def norm_vix(ts):
    v = ts_values(ts)
    return TimeSeries.from_times_and_values(ts.time_index, ((v - vix_mu)/vix_sd).astype(np.float32))

train_vix_z = norm_vix(train_vix)
val_vix_z   = norm_vix(val_vix)
test_vix_z  = norm_vix(test_vix)

full_vix_z = train_vix_z.concatenate(val_vix_z).concatenate(test_vix_z)

In [ ]:
import torch
from pytorch_lightning.callbacks import EarlyStopping
from darts.models import TCNModel

IN_LEN = 30
OUT_LEN = 1

early_stopper = EarlyStopping("val_loss", min_delta=1e-3, patience=5, verbose=True)

if torch.cuda.is_available():
    pl_trainer_kwargs = {"accelerator":"gpu","devices":1,"callbacks":[early_stopper],"num_sanity_val_steps":0,"gradient_clip_val":0.1}
    num_workers = 4
elif torch.backends.mps.is_available():
    pl_trainer_kwargs = {"accelerator":"mps","devices":1,"callbacks":[early_stopper],"num_sanity_val_steps":0,"gradient_clip_val":0.1}
    num_workers = 0
else:
    pl_trainer_kwargs = {"accelerator":"cpu","devices":1,"callbacks":[early_stopper],"num_sanity_val_steps":0,"gradient_clip_val":0.1}
    num_workers = 0

add_encoders = {"cyclic": {"past": ["dayofweek", "month"]}}

tcn = TCNModel(
    input_chunk_length=IN_LEN,
    output_chunk_length=OUT_LEN,
    batch_size=256,
    n_epochs=80,
    kernel_size=5,
    num_filters=8,
    dilation_base=2,
    dropout=0.2,
    optimizer_kwargs={"lr": 1e-3},
    add_encoders=add_encoders,
    pl_trainer_kwargs=pl_trainer_kwargs,
    model_name="tcn_vol_60B_to_1B_z",
    force_reset=True,
    save_checkpoints=True,
)

# longer val slice for early stopping
all_full = [tr.concatenate(va).concatenate(te) for tr,va,te in zip(train, val, test)]
val_len = len(val[0])
val_for_training = [s[-((2*val_len)+IN_LEN): -val_len] for s in all_full]

tcn.fit(
    series=train,
    val_series=val_for_training,
    max_samples_per_ts=4000,
    dataloader_kwargs={"num_workers": num_workers},
    verbose=True,
)

tcn = TCNModel.load_from_checkpoint("tcn_vol_60B_to_1B_z", best=True)
print("Loaded best checkpoint.")

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from darts.timeseries import concatenate

def _maybe_concat(x):
    return concatenate(x, axis=0) if isinstance(x, list) else x

def precompute_tcn_valtest_data_daily_fast(
    tcn,
    train,
    val,
    test,
    IN_LEN,
    R,
    vix_full_z=None,
    include_vix_now=True,
    pretest_keep=300,
    constant_eps=1e-8,
):
    """
    Faster version:
      - keep only a short pre-test context
      - use optimized historical_forecasts
      - batch over all series at once
      - build X with sliding_window_view
    """
    keep_pre = max(R, pretest_keep)

    full_list = []
    test_start_list = []

    for i in range(len(train)):
        tv = train[i].concatenate(val[i])
        ctx = tv[-(IN_LEN + keep_pre):]
        full = ctx.concatenate(test[i])
        full_list.append(full)
        test_start_list.append(test[i].start_time())

    # common start: first forecastable point after the history window
    start_ts = full_list[0].time_index[IN_LEN]

    preds_list = tcn.historical_forecasts(
        series=full_list,
        start=start_ts,
        forecast_horizon=1,
        stride=1,
        retrain=False,
        last_points_only=True,
        verbose=False,
        enable_optimization=True,
    )

    preds_list = [_maybe_concat(p) for p in preds_list]

    out = []
    for i in tqdm(range(len(full_list)), desc="assemble cache"):
        full = full_list[i]
        preds = preds_list[i]

        y = full.slice_intersect(preds)
        preds = preds.slice_intersect(y)

        yv = y.values(copy=False).squeeze(-1).astype(np.float32)
        pv = preds.values(copy=False).squeeze(-1).astype(np.float32)
        sv = np.abs(yv - pv).astype(np.float32)

        ts = y.time_index
        is_test = np.array([t >= test_start_list[i] for t in ts], dtype=bool)

        y_test = yv[is_test]
        const_test = (np.nanmax(y_test) - np.nanmin(y_test)) < constant_eps if y_test.size else True

        # vectorized raw lag window
        v = full.values(copy=False).squeeze(-1).astype(np.float32)
        vp = np.concatenate([np.full(IN_LEN, v[0], dtype=np.float32), v])
        win = sliding_window_view(vp, window_shape=IN_LEN)

        i0 = int(full.time_index.get_indexer([ts[0]])[0])
        X = win[i0:i0 + len(ts)].copy()

        if include_vix_now:
            if vix_full_z is None:
                raise ValueError("Need vix_full_z when include_vix_now=True")
            vix_short = vix_full_z.slice(full.start_time(), full.end_time())
            vix_aligned = vix_short.slice_intersect(y)
            v_now = vix_aligned.values(copy=False).squeeze(-1).astype(np.float32)
            X = np.concatenate([X, v_now[:, None]], axis=1)

        out.append({
            "series_id": i,
            "ts": ts,
            "y": yv,
            "pred": pv,
            "score": sv,
            "X": X,
            "is_test": is_test,
            "const_test": const_test,
        })

    return out

R = 200

data = precompute_tcn_valtest_data_daily_fast(
    tcn=tcn,
    train=train,
    val=val,
    test=test,
    IN_LEN=IN_LEN,
    R=R,
    vix_full_z=full_vix_z,
    include_vix_now=True,
    pretest_keep=300,
)

print("Built cache:", len(data), "series | X dim:", data[0]["X"].shape[1])

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from cp_methods import (
    eval_cp,
    eval_lcp,
    eval_aci,
    eval_dtaci,
    eval_olcp,
    eval_olcp_adahedge,
    eval_spci,
)

# =========================================================
# Config
# =========================================================
ALPHA = 0.1
BLOCK_LEN = 20          # ETF daily: 20; ELEC2: 48; ILINet: 26
B = 1000
SEED = 42

METHOD_ORDER = ["CP", "LCP", "ACI", "DTACI", "SPCI", "OLCP", "OLCPH"]

METHOD_RUNNERS = {
    "cp": eval_cp,
    "lcp": eval_lcp,
    "aci": eval_aci,
    "dtaci": eval_dtaci,
    "spci": lambda data, alpha, R, exclude_const_test=False, show_pbar=True: eval_spci(
        data,
        alpha=alpha,
        R=R,
        w_lag=30,          # ETF daily default; change to 24 for ELEC2, 8 for ILINet
        T_train=R,
        refit_every=5,     # ETF daily default; change to 24 for ELEC2, 1 for ILINet
        beta_grid_size=21,
        exclude_const_test=exclude_const_test,
        show_pbar=show_pbar,
    ),
    "olcp": eval_olcp,
    "olcph": eval_olcp_adahedge,
}


# =========================================================
# Width conversion
# =========================================================
def add_original_scale_width_if_possible(df, zstats=None):
    """
    If zstats is provided, convert z-score width to original scale.

    Adds:
      width_orig      = original abs-log-return width
      width_orig_pct  = 100 * original abs-log-return width

    If zstats is None, returns df unchanged.
    """
    d = df.copy()

    if zstats is None:
        return d

    sd = np.asarray([s for _, s in zstats], dtype=float)
    sid = d["series_id"].astype(int).to_numpy()

    width_z = (
        d["width"]
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=float)
    )

    d["width_orig"] = width_z * sd[sid]
    d["width_orig_pct"] = 100.0 * d["width_orig"]
    return d


def choose_width_col(df):
    """
    For ETF/VIX after z-score conversion, use width_orig_pct.
    Otherwise use raw width.
    """
    if "width_orig_pct" in df.columns:
        return "width_orig_pct"
    return "width"


# =========================================================
# Block bootstrap
# =========================================================
def _block_bootstrap_indices(n, block_len, rng):
    if n <= 0:
        return np.array([], dtype=int)

    block_len = int(min(max(1, block_len), n))
    n_blocks = int(np.ceil(n / block_len))
    max_start = n - block_len

    starts = rng.integers(0, max_start + 1, size=n_blocks)
    idx = np.concatenate([np.arange(s, s + block_len) for s in starts])
    return idx[:n]


def _nanmean_safe(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan


def bootstrap_compact_summary(
    df,
    method_name,
    width_col,
    B=1000,
    block_len=20,
    seed=42,
    show_pbar=True,
):
    """
    Returns one compact row:
      Method, Coverage, Coverage_SE, AvgSize, AvgSize_SE, Rows, B, block_len

    Coverage and AvgSize are first computed per series, then averaged over series.
    Bootstrap resamples time blocks within each series and recomputes that same
    mean-over-series statistic.
    """
    d = df.copy()
    d[width_col] = d[width_col].replace([np.inf, -np.inf], np.nan)

    # empirical per-series point estimates
    per = (
        d.groupby("series_id", observed=True)
        .agg(
            cov=("covered", "mean"),
            size=(width_col, "mean"),
            n=("covered", "size"),
        )
        .reset_index()
    )

    cov_hat = float(per["cov"].mean())
    size_hat = float(per["size"].mean())

    # arrays per series
    groups = []
    for sid, g in d.groupby("series_id", observed=True):
        g = g.sort_values("t") if "t" in g.columns else g.copy()
        groups.append({
            "covered": g["covered"].astype(float).to_numpy(),
            "size": g[width_col].astype(float).to_numpy(),
            "n": len(g),
        })

    rng = np.random.default_rng(seed)
    cov_boot = np.empty(B, dtype=float)
    size_boot = np.empty(B, dtype=float)

    iterator = range(B)
    if show_pbar:
        iterator = tqdm(iterator, desc=f"bootstrap {method_name}", leave=False)

    for b in iterator:
        cov_per_series = []
        size_per_series = []

        for gr in groups:
            idx = _block_bootstrap_indices(gr["n"], block_len, rng)

            cov_per_series.append(float(np.mean(gr["covered"][idx])))
            size_per_series.append(_nanmean_safe(gr["size"][idx]))

        cov_boot[b] = float(np.mean(cov_per_series))
        size_boot[b] = _nanmean_safe(size_per_series)

    cov_se = float(np.std(cov_boot[np.isfinite(cov_boot)], ddof=1))
    size_se = float(np.std(size_boot[np.isfinite(size_boot)], ddof=1))

    return {
        "Method": method_name,
        "Coverage": cov_hat,
        "Coverage_SE": cov_se,
        "AvgSize": size_hat,
        "AvgSize_SE": size_se,
        "Rows": int(len(d)),
        "B": int(B),
        "block_len": int(block_len),
    }


# =========================================================
# Run methods + compact bootstrap table
# =========================================================
results = {}
summary_rows = {}

# If zstats exists, use it. Otherwise raw width is used.
zstats_for_width = globals().get("zstats", None)

for i, (name, fn) in enumerate(METHOD_RUNNERS.items()):
    # Make stochastic OLCP-Hedge reproducible if it uses np.random internally
    np.random.seed(SEED + 1000 * i)

    df_m, sec = fn(
        data,
        alpha=ALPHA,
        R=R,
        exclude_const_test=False,
        show_pbar=True,
    )

    df_m = add_original_scale_width_if_possible(df_m, zstats=zstats_for_width)
    results[name] = df_m

    width_col = choose_width_col(df_m)

    row = bootstrap_compact_summary(
        df=df_m,
        method_name=name.upper(),
        width_col=width_col,
        B=B,
        block_len=BLOCK_LEN,
        seed=SEED + 1000 * i,
        show_pbar=True,
    )

    summary_rows[name] = row


paper_table = pd.DataFrame(summary_rows.values())

paper_table["Method"] = pd.Categorical(
    paper_table["Method"],
    categories=METHOD_ORDER,
    ordered=True,
)

paper_table = paper_table.sort_values("Method").reset_index(drop=True)

print(paper_table.to_string(index=False, formatters={
    "Coverage": "{:.3f}".format,
    "Coverage_SE": "{:.4f}".format,
    "AvgSize": "{:.4f}".format,
    "AvgSize_SE": "{:.4f}".format,
}))